# Create Point Cloud {#create_point_cloud_exercise}

Create a `pyvista.PolyData`{.interpreted-text role="class"} object from
a point cloud of vertices and scalar arrays for those points.


In [18]:
import numpy as np
import pyvista as pv
from pyvista import examples

Point clouds are generally constructed using
`pyvista.PolyData`{.interpreted-text role="class"} and can easily have
scalar or vector data arrays associated with the individual points. In
this example, we\'ll start by working backwards using a point cloud that
is available from our `examples` module. This however is no different
than creating a PyVista mesh with your own NumPy arrays of vertice
locations.


In [19]:
# Define some helpers - ignore these and use your own data if you like!
def generate_points(subset=0.02):
    """A helper to make a 3D NumPy array of points (n_points by 3)."""
    dataset = examples.download_lidar()
    ids = np.random.randint(low=0, high=dataset.n_points - 1, size=int(dataset.n_points * subset))
    return dataset.points[ids]


points = generate_points()
# Output the first 5 rows to prove it's a numpy array (n_points by 3)
# Columns are (X, Y, Z)
points[0:5, :]

pyvista_ndarray([[4.80979975e+05, 4.40022900e+06, 1.75808997e+03],
                 [4.81081875e+05, 4.40023980e+06, 1.75681995e+03],
                 [4.81004575e+05, 4.40018910e+06, 1.77195996e+03],
                 [4.80971575e+05, 4.40013640e+06, 1.76156006e+03],
                 [4.81011775e+05, 4.40014790e+06, 1.76115002e+03]])

Now that you have a NumPy array of points/vertices either from our
sample data or your own project, create a PyVista mesh using those
points.


In [20]:
# insert your code here
point_cloud = pv.PolyData(points)
point_cloud

PolyData (0x1ee80a3c460)
  N Cells:    67841
  N Points:   67841
  N Strips:   0
  X Bounds:   4.809e+05, 4.811e+05
  Y Bounds:   4.400e+06, 4.400e+06
  Z Bounds:   1.754e+03, 1.785e+03
  N Arrays:   0

Now, perform a sanity check to show that the points have been loaded
correctly.


In [21]:
np.allclose(points, point_cloud.points)

True

Now that we have a PyVista mesh, we can plot it. Note that we add an
option to use eye dome lighting - this is a shading technique to improve
depth perception with point clouds (learn more about
[EDL](https://docs.pyvista.org/examples/02-plot/edl.html)).


In [22]:
point_cloud.plot(eye_dome_lighting=True)

Widget(value='<iframe src="http://localhost:58201/index.html?ui=P_0x1ee80a77890_5&reconnect=auto" class="pyvis…

Now what if you have data attributes (scalar or vector arrays) that
you\'d like to associate with every point of your mesh? You can easily
add NumPy data arrays that have a length equal to the number of points
in the mesh along the first axis. For example, lets add a few arrays to
this new `point_cloud` mesh.

Make an array of scalar values with the same length as the points array.
Each element in this array will correspond to points at the same index:

::: note
::: title
Note
:::

You can use a component of the `points` array or use the `n_points`
property of the mesh to make an array of that length.
:::


In [23]:
data = point_cloud.points[:, 2]  # coordenada Z de cada punto
print(f"Shape: {data.shape}, n_points: {point_cloud.n_points}")

Shape: (67841,), n_points: 67841


Add that data to the mesh with the name \"elevation\".


In [24]:
# your code here
point_cloud.point_data["elevation"] = data
point_cloud

PolyData (0x1ee80a3c460)
  N Cells:    67841
  N Points:   67841
  N Strips:   0
  X Bounds:   4.809e+05, 4.811e+05
  Y Bounds:   4.400e+06, 4.400e+06
  Z Bounds:   1.754e+03, 1.785e+03
  N Arrays:   1

And now we can plot the point cloud with that elevation data. PyVista is
smart enough to plot the scalar array you added by default. This time,
let\'s render every point as its own sphere using
`render_points_as_spheres`.


In [25]:
point_cloud.plot(render_points_as_spheres=True)

Widget(value='<iframe src="http://localhost:58201/index.html?ui=P_0x1eede983890_6&reconnect=auto" class="pyvis…

That data is kind of boring, right? You can also add data arrays with
more than one scalar value - perhaps a vector with three elements?
Let\'s make a little function that will compute vectors for every point
in the point cloud and add those vectors to the mesh.

This time, we\'re going to create a totally new, random point cloud
containing 100 points using `numpy.random.random`{.interpreted-text
role="func"}.


In [26]:
# Create a random point cloud with Cartesian coordinates
points = np.random.rand(100, 3)
# Construct PolyData from those points
point_cloud = pv.PolyData(points)


def compute_vectors(mesh):
    """Create normalized vectors pointing outward from the center of the cloud."""
    origin = mesh.center
    vectors = mesh.points - origin
    return vectors / np.linalg.norm(vectors, axis=1)[:, None]


vectors = compute_vectors(point_cloud)
vectors[0:5, :]

pyvista_ndarray([[ 0.5232326 , -0.70845694,  0.47362054],
                 [ 0.19506669, -0.81422665,  0.54679425],
                 [-0.08041665, -0.6888954 , -0.72038621],
                 [-0.44715592,  0.88556772, -0.1257831 ],
                 [ 0.04542648,  0.6402127 ,  0.7668534 ]])

Add the vector array as point data to the new mesh:


In [27]:
point_cloud.point_data["vectors"] = vectors
point_cloud

PolyData (0x1ee80a3e920)
  N Cells:    100
  N Points:   100
  N Strips:   0
  X Bounds:   1.738e-02, 9.942e-01
  Y Bounds:   1.543e-02, 9.994e-01
  Z Bounds:   5.721e-04, 9.973e-01
  N Arrays:   1

Now we can make arrows using those vectors using the glyph filter (see
the [Glyph
Example](https://docs.pyvista.org/examples/01-filter/glyphs.html) for
more details).


In [28]:
arrows = point_cloud.glyph(
    orient="vectors",
    scale=False,
    factor=0.15,
)

# Display the arrows
plotter = pv.Plotter()
plotter.add_mesh(point_cloud, color="maroon", point_size=10.0, render_points_as_spheres=True)
plotter.add_mesh(arrows, color="lightblue")
# plotter.add_point_labels([point_cloud.center,], ['Center',],
#                          point_color='yellow', point_size=20)
plotter.show_grid()
plotter.show()

Widget(value='<iframe src="http://localhost:58201/index.html?ui=P_0x1ee80a77d90_7&reconnect=auto" class="pyvis…

```{=html}
<center>
  <a target="_blank" href="https://colab.research.google.com/github/pyvista/pyvista-tutorial/blob/gh-pages/notebooks/tutorial/02_mesh/exercises/b_create-point-cloud.ipynb">
    <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/ width="150px">
  </a>
</center>
```
